In [92]:
from __future__ import print_function
import warnings
import os
from python_speech_features import mfcc
from scipy.io import wavfile
from hmmlearn import hmm
import numpy as np

In [93]:
warnings.filterwarnings('ignore')

In [94]:
def extract_mfcc(full_audio_path):
    sample_rate, wave =  wavfile.read(full_audio_path)
    mfcc_features = mfcc(wave, samplerate=sample_rate)
    return mfcc_features

In [95]:
def buildDataSet(dir):
    # Filter out the wav audio files under the dir
    fileList = [f for f in os.listdir(dir) if os.path.splitext(f)[1] == '.wav']
    dataset = {}
    for fileName in fileList:
        tmp = fileName.split('.')[0]
        label = tmp.split('_')[1]
        feature = extract_mfcc(dir+fileName)
        if label not in dataset.keys():
            dataset[label] = []
            dataset[label].append(feature)
        else:
            exist_feature = dataset[label]
            exist_feature.append(feature)
            dataset[label] = exist_feature
    return dataset

In [96]:
def train_GMMHMM(dataset):
    GMMHMM_Models = {}
    states_num = 5

    for label in dataset.keys():
        model = hmm.GMMHMM(n_components=states_num, n_iter=10)
        trainData = dataset[label]
        length = np.zeros([len(trainData), ], dtype=np.int)
        for m in range(len(trainData)):
            length[m] = trainData[m].shape[0]
        trainData = np.vstack(trainData)
        model.fit(trainData, lengths=length)  # get optimal parameters
        GMMHMM_Models[label] = model
    return GMMHMM_Models

In [97]:
trainDir = './train_audio/'
trainDataSet = buildDataSet(trainDir)

In [98]:
trainDir = './train_audio/'
trainDataSet = buildDataSet(trainDir)
print("Finish prepare the training data")

hmmModels = train_GMMHMM(trainDataSet)
print("Finish training of the GMM_HMM models for digits 0-9")

Finish prepare the training data
Finish training of the GMM_HMM models for digits 0-9


In [99]:
print(hmmModels['1'].transmat_)

[[8.84861679e-001 3.83568951e-002 6.72578521e-096 3.22673787e-211
  7.67814255e-002]
 [4.63494193e-029 9.12755070e-001 8.72449300e-002 0.00000000e+000
  1.33902492e-020]
 [1.55609366e-056 2.61287395e-031 1.00000000e+000 9.30491298e-158
  3.13272547e-084]
 [7.71675531e-176 5.18406348e-192 3.07546812e-117 1.00000000e+000
  0.00000000e+000]
 [3.51190927e-025 1.24795024e-001 5.90287916e-108 0.00000000e+000
  8.75204976e-001]]


In [100]:
testDir = './test_audio/'
testDataSet = buildDataSet(testDir)

score_cnt = 0
for label in testDataSet.keys():
    feature = testDataSet[label]
    scoreList = {}
    for model_label in hmmModels.keys():
        model = hmmModels[model_label]
        score = model.score(feature[0])
        scoreList[model_label] = score
    predict = max(scoreList, key=scoreList.get)
    print("Test on true label ", label, ": predict result label is ", predict)
    if predict == label:
        score_cnt+=1
print("Final recognition rate is %.2f"%(100.0*score_cnt/len(testDataSet.keys())), "%")

Test on true label  9 : predict result label is  9
Test on true label  7 : predict result label is  7
Test on true label  8 : predict result label is  6
Test on true label  6 : predict result label is  6
Test on true label  1 : predict result label is  1
Test on true label  3 : predict result label is  3
Test on true label  5 : predict result label is  5
Test on true label  10 : predict result label is  9
Test on true label  2 : predict result label is  2
Test on true label  4 : predict result label is  4
Final recognition rate is 80.00 %
